In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Plotting_IQR import plot_distribution, get_training_data, plot_pfn_variance_surface, plot_GP_variance_surface
from pfn_evaluate import eval_pfn
import pfns4bo
from pfns4bo.scripts.acquisition_functions import TransformerBOMethod

/home/xic/sy493/.conda/envs/PFN_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load data
file_path = "results_LS_150/run_20260728_122058/metrics.pt"
metrics = torch.load(file_path)

file_path = "results_LS_150/run_20260728_122058/experimental_results.pt"
data = torch.load(file_path)

In [ ]:
#metrics = {
#        "pred_error": torch.zeros((n_tests, n_dims, n_methods, n_repeats, n_samples, n_fns)), # GP & PFN only
#        "total_error": torch.zeros((n_tests, n_dims, n_methods, n_repeats, 1)),
#        "EI": torch.zeros((n_tests, n_dims, n_methods, n_repeats, n_samples, 1))
#    }
pred_error = metrics["pred_error"]
total_error = metrics["total_error"]
ei = metrics["EI"]

y_true_store = data["y_true"][0]
mu_store = data["mu"][0]
var_store = data["var"][0]
x_queried = data["x_query"][0]

In [ ]:
# metrics["pred_error"][test, k, m_idx, rep, :, :]
# 9 tests, 1 dim, 2 methods, 21 reps, 1000 samples, dim = 1 -> 9 tests, 3 methods, 1000 samples, 1 
pred_error_med = torch.quantile(torch.sum(pred_error[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.5, dim=-2)
total_error_med = torch.quantile(pred_error[:, 0, :, :, :], 0.5, dim=-2)
ei_med = torch.quantile(torch.sum(ei[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.5, dim=-2)

pred_error_lq = torch.quantile(torch.sum(pred_error[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.25, dim=-2)
total_error_lq = torch.quantile(pred_error[:, 0, :, :, :], 0.25, dim=-2)
ei_lq = torch.quantile(torch.sum(ei[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.25, dim=-2)

pred_error_uq = torch.quantile(torch.sum(pred_error[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.75, dim=-2)
total_error_uq = torch.quantile(pred_error[:, 0, :, :, :], 0.75, dim=-2)
ei_uq = torch.quantile(torch.sum(ei[:, 0, :, :, :, :], dim=-1, keepdim=True), 0.75, dim=-2)

In [ ]:
mu_data_GP = torch.sum(mu_store[:, 0, 0, :, :], dim=-1, keepdim=True)
mu_data_PFN = torch.sum(mu_store[:, 1, 0, :, :], dim=-1, keepdim=True)
var_data_GP = torch.sum(var_store[:, 0, 0, :, :], dim=-1, keepdim=True)
var_data_PFN = torch.sum(var_store[:, 1, 0, :, :], dim=-1, keepdim=True)
x_query_GP = x_queried[:, 0, 0, :, :]
x_query_PFN = x_queried[:, 1, 0, :, :]

In [ ]:
for i in range(mu_data_GP.shape[0]):
    pred_ub_GP = mu_data_GP[i] + torch.sqrt(var_data_GP[i]) * 1
    pred_lb_GP = mu_data_GP[i] - torch.sqrt(var_data_GP[i]) * 1
    pred_ub_PFN = mu_data_PFN[i] + torch.sqrt(var_data_PFN[i]) * 1
    pred_lb_PFN = mu_data_PFN[i] - torch.sqrt(var_data_PFN[i]) * 1

    x_query_GP_plot = x_query_GP[i]
    x_query_PFN_plot = x_query_PFN[i]

    fig, (ax1, ax2) = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(8, 6),
        sharex=True,
        sharey=True,
    )
    ax1.fill_between(
        x_query_GP_plot,
        pred_lb_GP,
        pred_ub_GP,
        color="tab:blue",
        alpha=0.3,
        edgecolor="none",
        label="$\pm 1\sigma$ Band",
    )
    ax1.plot(x_query_GP_plot, mu_data_GP[i], color="tab:blue", linewidth=2, label="Mean A ($\mu_A$)")
    ax1.set_title("Series A")
    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.legend(loc="upper right")

    ax2.fill_between(
        x_query_PFN_plot,
        pred_lb_PFN,
        pred_ub_PFN,
        color="tab:orange",
        alpha=0.3,
        edgecolor="none",
        label="$\pm 1\sigma$ Band",
    )
    ax2.plot(x_query_PFN_plot, mu_data_PFN[i], color="tab:orange", linewidth=2, label="Mean B ($\mu_B$)")
    ax2.set_title("Series B")
    ax2.grid(True, linestyle="--", alpha=0.5)
    ax2.legend(loc="upper right")

    ax2.set_xlabel("Sequence Index / Time")
    fig.text(0.01, 0.5, "Value", va="center", rotation="vertical")

    plt.tight_layout()
    plt.show()